# FeynmanEngine for Undergraduate QFT — A Teaching Walk-through

**Audience**: physics teacher running a particle-physics or QFT course; undergraduate or early-graduate student following along.

**Goal**: see Feynman diagrams, the spin-averaged $|\overline{\mathcal{M}}|^2$, and the integrated cross-section side by side for the textbook examples — exactly the workflow you'd do by hand in Peskin & Schroeder Chapters 5–6.

**Prerequisite**: `pip install feynman-engine && feynman setup --skip-lhapdf --skip-openloops` is enough for everything in this notebook (no LHC physics here).

---

**Outline:**
1. Schwinger's first QED prediction: the anomalous magnetic moment $a_e = \alpha/2\pi$
2. $e^+ e^- \to \mu^+ \mu^-$ — the cleanest scattering process in the SM
3. Comparing the four classic 2→2 QED reactions (Bhabha, Møller, Compton, $e^+e^-\to\gamma\gamma$)
4. The angular shape: $1 + \cos^2\theta$ and where it comes from
5. The Z resonance — $e^+ e^- \to \mu^+\mu^-$ from QED to full electroweak
6. NLO in 30 seconds: the textbook K-factor $1 + 3\alpha/(4\pi)$

## 1. Schwinger 1948 — `a_e = α/(2π)`

The first prediction QED ever made beyond tree level: the anomalous magnetic moment of the electron equals $\alpha/2\pi \approx 0.001161$. Modern measurements agree with the full SM calculation to ~10 decimal places.

The engine evaluates this through the curated 1-loop QED vertex correction.

In [ ]:
import math
from fastapi.testclient import TestClient
from feynman_engine.api.app import app

client = TestClient(app)

ALPHA = 1.0 / 137.035999084
schwinger = ALPHA / (2 * math.pi)

r = client.get("/api/amplitude/loop-evaluate", params={
    "observable": "schwinger_amm", "q_sq": 1.0, "m_sq": 2.6e-7, "theory": "QED",
}).json()
engine_value = r.get("value_real") or r.get("value")

print(f"Engine value: a_e = {engine_value:.8f}")
print(f"Schwinger:    a_e = {schwinger:.8f}")
print(f"Difference: {abs(engine_value - schwinger):.2e}  (machine precision)")

## 2. `e⁺ e⁻ → μ⁺ μ⁻` — the textbook tree-level process

One $s$-channel photon. The integrated cross-section is
$$\sigma = \frac{4\pi\alpha^2}{3s}$$

Let's see (a) the diagram, (b) the $|\overline{\mathcal{M}}|^2$ formula, and (c) $\sigma$ at five energies a student might encounter:
- 5 GeV (BES-III)
- 10.58 GeV (Belle / BaBar Υ(4S))
- 29 GeV (PEP / PETRA)
- 91 GeV (LEP Z-pole)
- 200 GeV (LEP-2)

In [ ]:
# (a) The diagram
from feynman_engine.engine import FeynmanEngine
from IPython.display import SVG, display

engine = FeynmanEngine()
result = engine.generate("e+ e- -> mu+ mu-", theory="QED", loops=0, output_format="svg")
for d in result.diagrams:
    print(f"Diagram {d.id}: {d.topology}")
if result.images:
    display(SVG(list(result.images.values())[0]))

In [ ]:
# (b) The spin-averaged matrix element
from feynman_engine.physics.amplitude import get_amplitude

amp = get_amplitude("e+ e- -> mu+ mu-", "QED")
print(f"|M̄|² = {amp.msq}")
print(f"      = 2 e⁴ (t² + u²) / s²    (Peskin & Schroeder eq. 5.78)")

In [ ]:
# (c) σ vs √s — exact-textbook agreement
from feynman_engine.amplitudes.cross_section import total_cross_section

PB_GEV2 = 0.3894e9
print(f"  √s [GeV] | σ_engine [pb] | σ_textbook [pb] | agreement")
print(f"  ---------|---------------|-----------------|----------")
for sqrts, label in [(5.0, "BES-III"), (10.58, "Υ(4S)"), (29.0, "PEP/PETRA"),
                     (91.0, "LEP Z-peak"), (200.0, "LEP-2")]:
    sigma_textbook = 4 * math.pi * ALPHA**2 / (3 * sqrts**2) * PB_GEV2
    r = total_cross_section("e+ e- -> mu+ mu-", "QED", sqrt_s=sqrts)
    err = abs(r['sigma_pb'] - sigma_textbook) / sigma_textbook * 100
    print(f"  {sqrts:>7.2f}  | {r['sigma_pb']:>12.3f}  | {sigma_textbook:>14.3f}  | {err:.2f}%   {label}")

## 3. The four classic 2→2 QED reactions side by side

Every QFT course covers these:
- **$e^+ e^- \to \mu^+ \mu^-$** — only the $s$-channel photon
- **$e^+ e^- \to e^+ e^-$ (Bhabha)** — adds a $t$-channel photon (same final-state flavour)
- **$e^- e^- \to e^- e^-$ (Møller)** — only $t$- and $u$-channel (no $s$-channel without antiparticles)
- **$e^- \gamma \to e^- \gamma$ (Compton)** — $s$- and $u$-channel electron exchange

Same theory, different topologies, very different angular distributions.

In [ ]:
for proc in ["e+ e- -> mu+ mu-", "e+ e- -> e+ e-", "e- e- -> e- e-", "e- gamma -> e- gamma"]:
    res = engine.generate(proc, theory="QED", loops=0, output_format="tikz")
    n_diag = res.summary['total_diagrams']
    topologies = {d.topology for d in res.diagrams}
    print(f"  {proc:25s}  →  {n_diag} diagram(s), topologies = {topologies}")

## 4. The shape: $d\sigma / d\cos\theta \propto 1 + \cos^2\theta$

For $e^+e^- \to f\bar{f}$ via a vector mediator, the helicity amplitudes give the famous symmetric $1+\cos^2\theta$ angular distribution. Let's plot it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from feynman_engine.amplitudes.differential import differential_distribution

edges = np.linspace(-0.95, 0.95, 21)
hist = differential_distribution(
    "e+ e- -> mu+ mu-", "QED", sqrt_s=10.0,
    observable="cos_theta", bin_edges=edges,
)
centers = np.array(hist["bin_centers"])
ds_dc = np.array(hist["dsigma_dX_pb"])

# Theoretical 1 + cos²θ (rescaled to match)
shape = 1 + centers**2
shape_norm = ds_dc.sum() / shape.sum() * shape

plt.figure(figsize=(8, 4.5))
plt.bar(centers, ds_dc, width=0.085, color="steelblue", edgecolor="navy", label="engine")
plt.plot(centers, shape_norm, "r-", lw=2, label=r"theory: $1 + \cos^2\theta$ (normalised)")
plt.xlabel(r"$\cos\theta$")
plt.ylabel(r"$d\sigma/d\cos\theta$ [pb]")
plt.title(r"$e^+e^- \to \mu^+\mu^-$ at $\sqrt{s} = 10$ GeV")
plt.legend()
plt.tight_layout()
plt.show()

## 5. The Z resonance — same process, different theory tag

In **pure QED** the cross-section just falls as $1/s$. In the **full electroweak** theory (`theory="EW"`), the $s$-channel $Z$ adds a Breit-Wigner resonance that dwarfs the QED contribution at $\sqrt{s} \approx M_Z$.

Plotting both side by side shows the $\sim 100\times$ enhancement at the Z peak — the discovery channel for the Z at LEP.

In [ ]:
energies = np.linspace(50, 130, 30)
sigma_qed = []
sigma_ew = []
for e in energies:
    rq = total_cross_section("e+ e- -> mu+ mu-", "QED", sqrt_s=float(e))
    rw = total_cross_section("e+ e- -> mu+ mu-", "EW",  sqrt_s=float(e))
    sigma_qed.append(rq["sigma_pb"])
    sigma_ew.append(rw["sigma_pb"])

plt.figure(figsize=(8, 4.5))
plt.semilogy(energies, sigma_qed, "b-", lw=2, label="QED only (γ exchange)")
plt.semilogy(energies, sigma_ew,  "r-", lw=2, label="EW (γ + Z + interference)")
plt.axvline(91.1876, color="gray", ls="--", alpha=0.5, label=r"$M_Z = 91.19$ GeV")
plt.xlabel(r"$\sqrt{s}$ [GeV]")
plt.ylabel(r"$\sigma(e^+e^-\to\mu^+\mu^-)$ [pb]")
plt.title("Z resonance — discovery channel at LEP")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. NLO in 30 seconds — the textbook $K = 1 + 3\alpha/(4\pi)$

After tree level, the next-to-leading-order correction sums the photon vertex correction, vacuum polarization, box diagrams, and bremsstrahlung. A famous all-orders simplification gives:
$$\sigma_{NLO}(e^+e^- \to \mu^+\mu^-) = \sigma_{LO} \times \Big(1 + \frac{3\alpha}{4\pi}\Big)$$

The engine returns this exactly — to machine precision — for the QED textbook case.

In [ ]:
K_textbook = 1 + 3 * ALPHA / (4 * math.pi)

r = client.get("/api/amplitude/cross-section", params={
    "process": "e+ e- -> mu+ mu-", "theory": "QED", "sqrt_s": 91, "order": "NLO",
}).json()

print(f"Engine K_NLO = {r['k_factor']:.7f}")
print(f"Textbook     = {K_textbook:.7f}")
print(f"σ_NLO / σ_LO = {r['sigma_nlo_pb'] / r['sigma_born_pb']:.7f}")

## What's next for the classroom

- **Run the browser UI** during a lecture: `feynman serve`, then point students at http://localhost:8000. The sidebar has 120+ pre-loaded examples organised by theory (QED, QCD, EW, BSM).
- **Compare amplitudes by hand**: every process returns the symbolic $|\overline{\mathcal{M}}|^2$ — students can take it, plug in numbers, and re-derive the integrated cross-section as homework.
- **Extend to QCD**: the QCD theory tab covers `qq̄→gg`, `gg→gg` (3-gluon and 4-gluon vertices), `gg→tt̄` with full massive top, etc.
- **Loop-induced processes**: try $H \to \gamma\gamma$ at `loops=1` to see the W+top-loop integral that drove Higgs discovery.
- **Trust badges**: every result returns `validated` / `approximate` / `blocked` — a chance to talk about systematics in a calculation.